# Build Spark Session

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataRead")
    .master("local[*]")
    .getOrCreate()
)

spark

## Read Single CSV
### Syntax:
### spark.read.csv('Path')
### Options: header='true/false', schema='schema Definition',inferSchema='true/false', sep='delimiter', dateFormat='yyyy-MM-dd',mode='PERMISSIVE/DROPMALFORMED/FAILFAST' 
        

In [2]:
customer_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("C:/Users/navat/AzureDataBricks/Azure_Databricks/Raw_Source/Customer_Data/Amazon_Customer_Data_20210112.csv")
)

customer_df.show(truncate=False)

+----------+------------------+-----------------+------------------------------------+-----------------------------------+-------+-----------+---------------------------+--------------------+
|Customerid|Customer FirstName|Customer LastName|Customer Address1                   |Customer Address2                  |zipCode|CountryCode|Customer Email             |Customer PhoneNumber|
+----------+------------------+-----------------+------------------------------------+-----------------------------------+-------+-----------+---------------------------+--------------------+
|AMZ0000006|Emmanuel          |Desowja          |Mangalagiri Road, Chinakakani       |Mangalagiri , Guntur               |522436 |IND        |****sowja@gmail.com        |9032531***          |
|AMZ0000007|Nidish Kumar      |NK               |Old Club Road, Coastal Andhra Region|GV Thota, Guntur                   |522436 |IND        |****shmodala@gmail.com     |9948877***          |
|AMZ0000008|Anand Sagar       |Sivaji   

In [3]:
customer_df.printSchema()

root
 |-- Customerid: string (nullable = true)
 |-- Customer FirstName: string (nullable = true)
 |-- Customer LastName: string (nullable = true)
 |-- Customer Address1: string (nullable = true)
 |-- Customer Address2: string (nullable = true)
 |-- zipCode: string (nullable = true)
 |-- CountryCode: string (nullable = true)
 |-- Customer Email: string (nullable = true)
 |-- Customer PhoneNumber: string (nullable = true)



## Read Multiple CSV Files in single Folder

In [4]:
files = [
    r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Customer_Data\Amazon_Customer_Data_20210111.csv",
    r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Customer_Data\Amazon_Customer_Data_20210112.csv"
]

CustomerFiles_df = spark.read.csv(files, header=True, inferSchema=True)  

CustomerFiles_df.show(truncate=False)


+----------+------------------+-----------------+------------------------------------+-----------------------------------+-------+-----------+----------------------------+--------------------+
|Customerid|Customer FirstName|Customer LastName|Customer Address1                   |Customer Address2                  |zipCode|CountryCode|Customer Email              |Customer PhoneNumber|
+----------+------------------+-----------------+------------------------------------+-----------------------------------+-------+-----------+----------------------------+--------------------+
|AMZ0000006|Emmanuel          |Desowja          |Mangalagiri Road, Chinakakani       |Mangalagiri , Guntur               |522436 |IND        |****sowja@gmail.com         |9032531***          |
|AMZ0000007|Nidish Kumar      |NK               |Old Club Road, Coastal Andhra Region|GV Thota, Guntur                   |522436 |IND        |****shmodala@gmail.com      |9948877***          |
|AMZ0000008|Anand Sagar       |Siva

In [5]:
#Attaching with Source File Name
from pyspark.sql.functions import input_file_name

files = [
    r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Customer_Data\Amazon_Customer_Data_20210111.csv",
    r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Customer_Data\Amazon_Customer_Data_20210112.csv"
]

CustomerFiles_df = spark.read.csv(files, header=True, inferSchema=True).withColumn("source_file", input_file_name())

CustomerFiles_df.show(truncate=False)

+----------+------------------+-----------------+------------------------------------+-----------------------------------+-------+-----------+----------------------------+--------------------+------------------------------------------------------------------------------------------------------------------+
|Customerid|Customer FirstName|Customer LastName|Customer Address1                   |Customer Address2                  |zipCode|CountryCode|Customer Email              |Customer PhoneNumber|source_file                                                                                                       |
+----------+------------------+-----------------+------------------------------------+-----------------------------------+-------+-----------+----------------------------+--------------------+------------------------------------------------------------------------------------------------------------------+
|AMZ0000006|Emmanuel          |Desowja          |Mangalagiri Road, Chinakaka

## Reading FixedWidth File (with Text)

In [20]:
from pyspark.sql.functions import substring as substr,trim

CustomerFixedFiles_df = (
    spark.read.text(r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Customer_Data\Myntra_Customers_Data_20210112.dat")
    .withColumn("CustomerId", trim(substr("value", 1, 11)))
    .withColumn("CustomerName", trim(substr("value", 12, 29)))
    .withColumn("CustomerAddress1", trim(substr("value", 41, 39)))
    .withColumn("CustomerAddress2", trim(substr("value", 80, 36)))
    .withColumn("CustomerZipCode", trim(substr("value", 116, 8)))
    .withColumn("CustomerCountryCode", trim(substr("value", 124, 4)))
    .withColumn("CustomerMobileNumber", trim(substr("value", 128, 11)))
    .withColumn("CustomerEmailAddress", trim(substr("value", 139, 29)))
    .drop("value")
)

CustomerFixedFiles_df.show(truncate=False)

+----------+--------------------------+------------------------------------+-----------------------------------+---------------+-------------------+--------------------+---------------------------+
|CustomerId|CustomerName              |CustomerAddress1                    |CustomerAddress2                   |CustomerZipCode|CustomerCountryCode|CustomerMobileNumber|CustomerEmailAddress       |
+----------+--------------------------+------------------------------------+-----------------------------------+---------------+-------------------+--------------------+---------------------------+
|MYNT00001 |Rayudu Lakshmi            |Door No 39d/3, 8th Lane             |Kotha Peta, Guntur                 |522436         |IND                |9160044***          |****hmirayudu.49@gmail.com |
|MYNT00002 |Gummadi Sri Sai Nikhil    |1st Line, Ramireddy Thota           |Kothapet, Guntur                   |522436         |IND                |9490320***          |****thethriller@gmail.com  |
|MYNT00003

# Read Excel Files with Sheet Names

### using Pandas to Spark DF

In [5]:
import pandas as pd

file_path = r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Customer_Data\FlipKart_Customer_Data_20210112.xlsx"

excelCustomer_df = pd.read_excel(file_path,sheet_name="Sheet1")

df = spark.createDataFrame(excelCustomer_df)

df.show(truncate=False)


+----------+------------------------+------------------------------------+-----------------------------------+-------+-----------+--------------------------------+--------------------+
|Customerid|CustomerName            |Customer Address1                   |Customer Address2                  |zipCode|CountryCode|Customer Email                  |Customer PhoneNumber|
+----------+------------------------+------------------------------------+-----------------------------------+-------+-----------+--------------------------------+--------------------+
|FLK00001  |Pedapati Srinh          |Door No 39d/3, 8th Lane             |Kotha Peta, Guntur                 |522436 |IND        |****upedapati50@gmail.com       |9000531***          |
|FLK00002  |P.V.S.Sreekanth         |1st Line, Ramireddy Thota           |Kothapet, Guntur                   |522436 |IND        |****reekanth.sreekanth@gmail.com|7416194***          |
|FLK00003  |Mohan Krishna           |214/9, Station Road                 |R

# Read XML Files

In [10]:
xmlfile=r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Customer_Data\Snap_Customer_Data_20210112.xml"

df = spark.read \
    .format('com.databricks.spark.xml') \
    .options(rowTag='row') \
    .load(xmlfile)

df.show()

+-----------+--------------------+--------------------+--------------------+--------------------+-------------------+----------+-------+
|CountryCode|    CustomerAddress1|    CustomerAddress2|       CustomerEmail|        CustomerName|CustomerPhoneNumber|Customerid|zipCode|
+-----------+--------------------+--------------------+--------------------+--------------------+-------------------+----------+-------+
|        IND|Door No 39d/3, 8t...|  Kotha Peta, Guntur|****umar23419@gma...|      Madda Ramkumar|         9949812***|  SNAP0001| 522436|
|        IND|1st Line, Ramired...|    Kothapet, Guntur| ****yaa25@yahoo.com|     Aditya Srinivas|         9493891***|  SNAP0002| 522436|
|        IND| 214/9, Station Road|     Railpet, Guntur|****ram2493@gmail...|  Pasalapudi Sai Ram|         7207601***|  SNAP0003| 522436|
|        IND|      6/2 Arundelpet|              Guntur|****kaleem94@gmai...|    Syed Kaleemuddin|         9492084***|  SNAP0004| 522436|
|        IND|       Old Club Road|    Kot

#  Read JSON File   

In [12]:
jsonFile=r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Customer_Data\Ajio_Customer_Data_20210112.json"

json_df = spark.read \
    .option("multiline", "true") \
    .json(jsonFile)

json_df.show()

+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+-------+
|CountryCode|   Customer Address1|   Customer Address2|      Customer Email|       Customer Name|Customer PhoneNumber|Customerid|zipCode|
+-----------+--------------------+--------------------+--------------------+--------------------+--------------------+----------+-------+
|        IND|Door No 39d/3, 8t...|  Kotha Peta, Guntur|****wanth.tadi470...|     Yaswanth Sriram|          9491489***|   AJ00001| 522436|
|        IND|1st Line, Ramired...|    Kothapet, Guntur|****allidi53@gmai...|Venkata Surendra ...|          9959335***|   AJ00002| 522436|
|        IND| 214/9, Station Road|     Railpet, Guntur|****ry51435@gmail...|Prudhvi Charan Jy...|          9014311***|   AJ00003| 522436|
|        IND|      6/2 Arundelpet|              Guntur|****iran3255@gmai...|Vakkalagadda Saik...|          9700053***|   AJ00004| 522436|
|        IND|       Old Club Road|

### with JSON Explode

In [ ]:
from pyspark.sql.functions import explode, col

jsonFile=r"C:\Users\navat\AzureDataBricks\Azure_Databricks\Raw_Source\Customer_Data\TataCliQ_Customer_Data_YYYYMMDD.json"

json_df = spark.read \
    .option("multiline", "true") \
    .json(jsonFile)

json_df.show(truncate=False)

df_exploded = json_df \
    .withColumn("CustomerEmail", col("Communication").getField("Customer Email")) \
    .withColumn("CustomerPhoneNumber", col("Communication").getField("Customer PhoneNumber")) \
    .withColumn("FirstName", col("Name").getField("First Name")) \
    .withColumn("LastName", col("Name").getField("Last Name")) \
    .drop("Communication", "Name")

df_exploded.show(truncate=False)



+-----------------------------------------+-----------+-----------------------+------------------+----------+------------------+-------+
|Communication                            |CountryCode|Customer Address1      |Customer Address2 |Customerid|Name              |zipCode|
+-----------------------------------------+-----------+-----------------------+------------------+----------+------------------+-------+
|{****wanth.tadi470@gmail.com, 9491489***}|IND        |Door No 39d/3, 8th Lane|Kotha Peta, Guntur|TTC00001  |{Yaswanth, Sriram}|522436 |
|{****athak@gmail.com, 8332830***}        |IND        |Old Club Road          |Kothapet, Guntur  |TTC00002  |{Puskar, Pathak}  |522436 |
+-----------------------------------------+-----------+-----------------------+------------------+----------+------------------+-------+



{"ts": "2026-07-02 14:07:54.281", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"explode(Communication)\" due to data type mismatch: The first parameter requires the (\"ARRAY\" or \"MAP\") type, however \"Communication\" has the type \"STRUCT<`Customer Email`: STRING, `Customer PhoneNumber`: STRING>\". SQLSTATE: 42K09", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "explode", "errorClass": "DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o152.withColumn.\n: org.apache.spark.sql.AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve \"explode(Communication)\" due to data type mismatch: The first parameter requires the (\"ARRAY\" or \"MAP\") type, however \"Communication\" has the type \"STRUCT<`Customer Email`: STRING, `Customer PhoneNu

AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "explode(Communication)" due to data type mismatch: The first parameter requires the ("ARRAY" or "MAP") type, however "Communication" has the type "STRUCT<`Customer Email`: STRING, `Customer PhoneNumber`: STRING>". SQLSTATE: 42K09;
'Project [Communication#243, CountryCode#244, Customer Address1#245, Customer Address2#246, Customerid#247, Name#248, zipCode#249, explode(Communication#243) AS comExplode#280]
+- Relation [Communication#243,CountryCode#244,Customer Address1#245,Customer Address2#246,Customerid#247,Name#248,zipCode#249] json
